1. 스프레드시트 만들어서 모든 필드에 대한 상세 설명 만들기(ppt에 사용)
2. 마켓 리서치, 서비스 리서치, 논문등을 통해 데이터 분석에 사용하는 llm 교정과 관련된 kpi 지표 찾기. 프로젝트 스토리는 그렇게 만들어야 함.
3. 

In [ ]:
#!pip install koreanize-matplotlib

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import platform
import koreanize_matplotlib

In [ ]:
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 1. 사용자별 사용량 데이터

2025년 6월 1일에서부터 10월 22일까지의 사용자별 서비스 사용량 집계 데이터입니다.
각 사용자의 고유 ID와 함께 총 사용 횟수, 그리고 최근 실행 날짜가 포함되어 있습니다.

##### 주요 필드
1. 2_id: 사용자 고유 식별자 (MongoDB ObjectId 형식)
2. count: 해당 사용자의 총 사용 횟수
3. recent_execution_date: 가장 최근 사용 날짜

In [ ]:
df1 = pd.read_json('data/1. 사용자별 사용량 데이터.json')

In [ ]:
df1.describe()

In [ ]:
df1['count'].describe()

In [ ]:
print(f"데이터 형태: {df1.shape}")


In [ ]:
df1.columns

In [ ]:
df1.info()

### 전처리 - id 데이터

In [ ]:
df1['_id'].head()

In [ ]:
df1['_id'] = df1['_id'].apply(lambda x: x.get('$oid') if isinstance(x, dict) else None)

In [ ]:
df1['_id'].head()

In [ ]:
df1['_id'].head()

### 전처리 - 사용 시간대 분리

In [ ]:
df1['exec_datetime'] = pd.to_datetime(
    df1['recent_execution_date'].apply(lambda x: x.get('$date') if isinstance(x, dict) else None)
)
df1 = df1.drop('recent_execution_date', axis=1)

In [ ]:
print("\n--- 'exec_datetime' 컬럼의 데이터 타입 ---")
print(df1['exec_datetime'].dtype)
df1['exec_datetime']

### 데이터 탐색 - 사용량 분포 확인

In [ ]:
df1['count'].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(data= df1, x='count', kde=True, log_scale=True, ax=ax)
ax.set_title("사용량 분포")
ax.set_xlabel('사용량')
ax.set_ylabel('Count')

ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.xaxis.set_minor_formatter(ticker.NullFormatter())

plt.show()

소수의 사용자가 엄청나게 많은 사용량을 보이며, 평균을 크게 끌어올리는 모습을 보여주고 있다. 상위 백분위수를 확인하여 더 자세히 살펴보자.

In [ ]:
q75 = df1['count'].quantile(0.75)
print(f"상위 25% (75th percentile) 기준: {q75:.2f} 건")

q90 = df1['count'].quantile(0.90)
print(f"상위 10% (90th percentile) 기준: {q90:.2f} 건")

q95 = df1['count'].quantile(0.95)
print(f"상위 5% (95th percentile) 기준: {q95:.2f} 건")

q99 = df1['count'].quantile(0.99)
print(f"상위 1% (99th percentile) 기준: {q99:.2f} 건")

In [ ]:
df1_sorted = df1.sort_values(by='count', ascending=False)

# 2. 총 사용량 계산
total_usage = df1_sorted['count'].sum()

# 3. 누적 사용량 및 비율 계산
df1_sorted['cumulative_usage'] = df1_sorted['count'].cumsum()
df1_sorted['cumulative_usage_ratio'] = df1_sorted['cumulative_usage'] / total_usage

# 4. 전체 사용량의 80%가 되는 지점 확인
# (80%를 넘지 않는 마지막 사용자 그룹)
pareto_group = df1_sorted[df1_sorted['cumulative_usage_ratio'] <= 0.80]

# 5. 결과 출력
user_count_total = len(df1)
user_count_pareto = len(pareto_group)
min_usage_pareto = pareto_group['count'].min()

print(f"--- 파레토 분석 (80%) ---")
print(f"총 사용자: {user_count_total}명 (총 사용량: {total_usage})")
print(f"전체 사용량의 80%를 차지하는 사용자 수: {user_count_pareto}명")
print(f"이들은 전체 사용자의 상위 {user_count_pareto / user_count_total * 100:.2f}% 입니다.")
print(f"이 그룹의 최소 사용량 (기준점): {min_usage_pareto} 건")

### 데이터 탐색 - 가장 최근 사용 날짜 확인

In [ ]:
min_time = df1['exec_datetime'].min()
max_time = df1['exec_datetime'].max()
print(f"최솟값 (가장 이른 시간): {min_time}")
print(f"최댓값 (가장 늦은 시간): {max_time}")

시간은 2025-10-22 12:04:32로 고정되어 있다.
즉, 'Count' 데이터는 2025년 6월 1일에서부터 10월 22일까지의 사용자의 누적합임을 알 수 있다.

# 2. 사용자 클라이언트 데이터

웹 기반 사용자 분석 데이터로, Mixpanel을 통해 수집한 사용자의 클라이언트 데이터입니다.

#### 데이터 구조 설명

##### 사용자 식별자 (distinct_id) 구분
1. 비로그인 사용자 (\\$device: 접두사)
-- \\$device:로 시작하는 식별자
-- 로그인하지 않은 사용자의 디바이스 기반 추적
-- 가입 유저와 미가입 유저가 모두 포함될 수 있음
2. 로그인 사용자 (MongoDB ObjectId 형식)
-- \\$device:가 아닌 숫자와 알파벳 조합
-- MongoDB ObjectId 형식의 사용자 고유 식별자
-- 실제 계정에 로그인한 사용자만 해당
##### 속성 정보 (properties)
 - 지리적 정보: 국가(country_code), 지역(region), 도시(city), 시간대(timezone)
 - 기술적 정보: 브라우저(browser), 브라우저 버전(browser_version), 운영체제(os)
 - 트래픽 소스: 초기 참조자(initial_referrer), 참조 도메인(initial_referring_domain)
 - 마케팅 정보: UTM 캠페인 정보 (initial_utm_campaign, initial_utm_source 등)
 - 계정 정보: 소셜 로그인 여부 (\\$is_google_account, \\$is_kakao_account 등)
 - 활동 정보: 마지막 방문 시간(last_seen)

In [ ]:
import pandas as pd
from pathlib import Path
import json

data_list2 = []
data2_path = r'C:\Workspace\99-archive\bookend\bookend-yeardream-proj\data\raw\2. 사용자 클라이언트 데이터.json'

with open(Path(data2_path), 'r', encoding='utf-8') as f:
    for line in f:
        data_list2.append(json.loads(line))
df2 = pd.json_normalize(data_list2)

In [ ]:
print(f"데이터 형태: {df2.shape}")
df2.head(10)

In [ ]:
total_rows = len(df2)
unique_ids = df2['distinct_id'].nunique()

print(f"전체 행 개수: {total_rows}")
print(f"유니크 ID 개수: {unique_ids}")
print(f"ID당 평균 기록 횟수: {total_rows / unique_ids:.2f}")

In [ ]:
df2.columns

In [ ]:
df2.info()

### 중복되는 'distinct_id'와 'properties.distinct_id'를 확인하기

In [ ]:
are_equal = df2['distinct_id'].equals(df2['properties.distinct_id'])

if are_equal:
    print("두 컬럼은 100% 완벽하게 일치합니다.")
else:
    print("두 컬럼은 서로 다른 부분이 있습니다.")

In [ ]:
# 중복되는 컬럼 지우기
df2 = df2.drop(columns='properties.distinct_id')

### 전처리 - 컬럼에서 'properties' 지우기

In [ ]:
df2.columns = df2.columns.str.replace('properties.','', regex=False)

### Feature Engineering - distinct_id

In [ ]:
df2['distinct_id'].head()

머신러닝 모델이 '로그인 사용자'와 '비로그인 사용자'를 구분하여 학습하도록 'is_logged_in'이라는 새로운 컬럼 만들기

In [ ]:
condition = df2['distinct_id'].str.startswith('$device:', na=True)
df2['is_logged_in'] = np.where(condition, 0, 1)

# 컬럼 순서 옮기기
cols = ['is_logged_in'] + [col for col in df2.columns if col != 'is_logged_in']
df2 = df2[cols]

전처리 - 'distinct_id'에서 \\$device 태그 지우기

In [ ]:
df2['distinct_id'] = df2['distinct_id'].apply(
    lambda x: x.split(':')[-1] if isinstance(x, str) and x.startswith('$device') else x
)

### 데이터 탐색 - 로그인 사용자와 비로그인 사용자 시각화

In [ ]:
user = df2.groupby('is_logged_in').count()

해당 데이터에는 총 6695명의 사용자가 있다.

In [ ]:
data_for_pie = user['distinct_id']
labels = ['비로그인 사용자', '로그인 사용자']
sns.set_style('whitegrid', {'font.family': 'Malgun Gothic'})
plt.figure(figsize=(8, 8))

plt.pie(data_for_pie,
        labels=labels,
        autopct='%1.1f%%',  #각 항목의 백분율을 소수점 첫째 자리까지 표시
        startangle=90,      # 차트가 시작되는 각도 설정
        colors=sns.color_palette('pastel'), # Seaborn의 파스텔 색상 팔레트 사용
        wedgeprops={'edgecolor': 'white', 'linewidth': 1}) # 파이 조각 테두리 설정

plt.title('로그인 사용자 분포', fontsize=16)
plt.ylabel('') # 불필요한 y-axis 라벨 제거
plt.axis('equal') # 원형을 유지하도록 설정
plt.show()

In [ ]:
df2['properties.browser_version'].value_counts()

In [ ]:
df2['properties.email'].head()

In [ ]:
df2['properties.geo_source'].head()

In [ ]:
df2['properties.name'].value_counts()

In [ ]:
df2['properties.$role'].value_counts()

# 3. 이벤트 데이터

#### 데이터 구조 설명

##### 주요 이벤트 유형
- `pageview_ad inflow`: 광고 유입
- `editor_run_paraphrasing`: 웹 에디터에서 문장 교정 실행
- `editor_selected_paraphrasing`: 웹 에디터에서 문장 교정 결과 중 하나를 선택하여 적용
- `run_paraphrasing`: 문장 교정 기능 실행
- `selected_paraphrasing`: 문장 교정 결과 중 하나를 선택하여 적용
- `open_sidepanel`: 사이드 패널 열기(많은 교정 작업에 용이)
- `tutorial_start`: 튜토리얼 시작
- `tutorial_complete`: 튜토리얼 완료

##### 공통 기본 속성 (모든 이벤트에 포함)
- 사용자 식별 정보:
-- `distinct_id`: 사용자 고유 식별자
-- `device_id`: 디바이스 고유 식별자
-- `user_id`: 로그인한 사용자 ID (null일 수 있음)
-- `insert_id`: 이벤트 고유 식별자

- 시간 정보:
-- `time`: 초 단위 Unix 타임스탬프
-- `mp_api_timestamp_ms`: Mixpanel API 타임스탬프 (밀리초)
-- `mp_processing_time_ms`: 처리 시간 (밀리초)

- 브라우저/디바이스 정보:
-- `browser`: 브라우저 종류 (Chrome, Whale Browser, Microsoft Edge 등)
-- `browser_version`: 브라우저 버전
-- `os`: 운영체제 (Windows, Mac OS X, Android 등)
-- `device`: 디바이스 타입 (Android, null 등)
-- `screen_height`, `screen_width`: 화면 해상도

- 지역 정보:
-- `city`: 도시 (Goyang-si, Seoul, Daejeon 등)
-- `region`: 지역 (Gyeonggi-do, Seoul 등)
-- `mp_country_code`: 국가 코드 (KR)

- 웹사이트 정보:
-- `current_url`: 현재 페이지 URL
-- `referrer`: 리퍼럴 URL
-- `referring_domain`: 리퍼럴 도메인
-- `initial_referrer`: 초기 리퍼럴
-- `initial_referring_domain`: 초기 리퍼럴 도메인
-- `search_engine`: 검색 엔진 (google 등)

- 기술적 정보:
-- `lib_version`: Mixpanel 라이브러리 버전
-- `mp_api_endpoint`: API 엔드포인트
-- `mp_lib`: 라이브러리 타입 (web)
-- `mp_loader`: 로더 타입 (gtm, null)
-- `environment`: 환경 (production)
-- `extension_version`: 확장 프로그램 버전

#### 이벤트별 특화 속성
##### `run_paraphrasing` 이벤트
1) AI 처리 관련:
- `llm_name` : AI 모델명
- `llm_provider` : AI 제공업체
- `llm_version` : AI 모델 버전
- `maintenance` (교정 강도) : 교정 강도수준
- `target_language` (교정 도착 언어) : 실행 결과로 노출되는 언어
- `tone` (제거됨)
- `terminal_word` (제거됨)

2) 사용자 입력:
- `input_sentence_length` : 입력 문장 길이
- `field` : 교정 목적 카테고리 (thesis, article, marketing, email, none
등)

3) UI/UX 관련:
-`position` : 사용 위치  (sidepanel, modal)
-`platform` : 데스크톱 앱 여부
-`trigger` : 트리거 방식  (floating_icon, bottom_icon_tab 등)

4) 성능 측정: 교정 결과가 도착하기까지의 시간
-`response_time_ms` (응답 시간 ms)
-`response_time_seconds` (응답 시간 sec)

##### `selected_paraphrasing` 이벤트 (`run_paraphrasing`과 동일한 속성 포함)
1) 교정 선택 관련:
-`index` : 3개의 제시 문장 중 선택된 결과의 인덱스 (1-N)
-`selected_sentence_id` : 선택 문장 ID
-`total_paraphrasing_sentence_count` : 동일 문장으로 재실행한 수(3의 배수로
측정: 3-1회 실행, 6-2회 실행)



##### `pageview_ad inflow` 이벤트
1) 광고 관련: `gclid` (Google 광고 클릭 ID), `medium` (cpc), `source` (google)

##### `open_sidepanel` 이벤트
1) `trigger`: 트리거 방식 (toggle_drawer_icon: 모달 내에 있는 사이드바 아이콘으로 실행,
bottom_icon_tab: 화면 우 하단에 노출하는 아이콘으로 실행)

##### `pageview_ad inflow` 이벤트
1) 광고 관련: `gclid` (Google 광고 클릭 ID), `medium` (cpc), `source` (google)

##### `open_sidepanel` 이벤트
1) `trigger`: 트리거 방식 (toggle_drawer_icon: 모달 내에 있는 사이드바 아이콘으로 실행,
bottom_icon_tab: 화면 우 하단에 노출하는 아이콘으로 실행)#### 브라우저별 특화 속성
- `browser_is_whale`: Whale 브라우저 여부
- `browser_type`: 브라우저 타입 (whale)
- `browser_user_agent`: 상세 User Agent
- `browser_is_chrome`: Chrome 브라우저 여부
- `browser_is_edge`: Edge 브라우저 여부

#### 로그인 상태 관련
- `is_logged_in`: 로그인 여부
- `email`: 이메일 주소
- `name`: 사용자 닉네임

In [ ]:
import pandas as pd
from pathlib import Path
import json

data_list3 = []
data3_path = r'C:\Workspace\99-archive\bookend\bookend-yeardream-proj\data\raw\3. 이벤트 데이터.json'

with open(Path(data3_path), 'r', encoding='utf-8') as f:
    for line in f:
        data_list3.append(json.loads(line))

df3 = pd.json_normalize(data_list3)

In [ ]:
print(f"데이터 형태: {df3.shape}")
df3.head(10)

In [ ]:
total_rows = len(df3)
unique_ids = df3['distinct_id'].nunique()

print(f"전체 행 개수: {total_rows}")
print(f"유니크 ID 개수: {unique_ids}")
print(f"ID당 평균 기록 횟수: {total_rows / unique_ids:.2f}")

In [ ]:
df3.columns

In [ ]:
df3.info()

### 중복되는 컬럼을 확인하기

In [ ]:
df3 = df3.fillna('null').replace("", 'null')

## 제거할 컬럼

In [ ]:
df3 = df3.drop(columns=[
    'properties.insert_id',
    'properties.device_id',
    'properties.user_id',         # id 중복
    'properties.screen_height',   
    'properties.screen_width',    # 해상도 중복
    'properties.browser_version', # 브라우저 버전
    'properties.search_engine',   # 4만건 이상이 결측치, 나머지 대부분은 google
    'properties.lib_version',     # Mixpanel 버전
    'properties.mp_api_endpoint', # 모두 같음
    'properties.mp_lib', 
    'properties.mp_loader',       # 결측치 5만건  
    'properties.extension_version']) # 확장 프로그램 버전

In [ ]:
are_equal = df3['insert_id'].equals(df3['properties.insert_id'])

if are_equal:
    print("두 컬럼은 100% 완벽하게 일치합니다.")
else:
    print("두 컬럼은 서로 다른 부분이 있습니다.")

In [ ]:
are_equal = df3['device_id'].equals(df3['properties.device_id'])

if are_equal:
    print("두 컬럼은 100% 완벽하게 일치합니다.")
else:
    print("두 컬럼은 서로 다른 부분이 있습니다.")

In [ ]:
are_equal = df3['user_id'].equals(df3['properties.user_id'])

if are_equal:
    print("두 컬럼은 100% 완벽하게 일치합니다.")
else:
    print("두 컬럼은 서로 다른 부분이 있습니다.")

In [ ]:
df3['event_name'].value_counts()

- 브라우저/디바이스 정보:
-- `browser`: 브라우저 종류 (Chrome, Whale Browser, Microsoft Edge 등)
-- `browser_version`: 브라우저 버전
-- `os`: 운영체제 (Windows, Mac OS X, Android 등)
-- `device`: 디바이스 타입 (Android, null 등)
-- `screen_height`, `screen_width`: 화면 해상도

In [ ]:
# -- `browser`: 브라우저 종류 (Chrome, Whale Browser, Microsoft Edge 등)
df3['properties.browser'].value_counts()

In [ ]:
# -- `browser_version`: 브라우저 버전
df3['properties.browser_version'].value_counts()

In [ ]:
# -- `os`: 운영체제 (Windows, Mac OS X, Android 등)
df3['properties.os'].value_counts()

In [ ]:
# -- `device`: 디바이스 타입 (Android, null 등)
df3['properties.device'].value_counts()

In [ ]:
# -- `screen_height`, `screen_width`: 화면 해상도
df3['properties.screen_height'].value_counts()

In [ ]:
df3['properties.screen_width'].value_counts()

### 시간 정보 컬럼 확인하기
(time, properties.mp_api_timestamp_ms , properties.mp_processing_time_ms)

In [ ]:
df3['time']

In [ ]:
df3['properties.city'].value_counts()

In [ ]:
df3['properties.region'].value_counts()

In [ ]:
df3['properties.mp_country_code'].value_counts()

In [ ]:
df3['properties.mp_api_timestamp_ms']

In [ ]:
df3['properties.mp_processing_time_ms']

### 웹사이트 정보
-- `current_url`: 현재 페이지 URL
-- `referrer`: 리퍼럴 URL
-- `referring_domain`: 리퍼럴 도메인
-- `initial_referrer`: 초기 리퍼럴
-- `initial_referring_domain`: 초기 리퍼럴 도메인
-- `search_engine`: 검색 엔진 (google 등)

In [ ]:
df3['properties.current_url'].value_counts()

In [ ]:
df3['properties.referrer'].value_counts()

In [ ]:
df3['properties.referring_domain'].value_counts()

In [ ]:
df3['properties.initial_referrer'].value_counts()

In [ ]:
df3['properties.initial_referring_domain'].value_counts()

In [ ]:
df3['properties.search_engine'].value_counts()

기술적 정보
-- `lib_version`: Mixpanel 라이브러리 버전
-- `mp_api_endpoint`: API 엔드포인트
-- `mp_lib`: 라이브러리 타입 (web)
-- `mp_loader`: 로더 타입 (gtm, null)
-- `environment`: 환경 (production)
-- `extension_version`: 확장 프로그램 버전

In [ ]:
df3['properties.lib_version'].value_counts()

In [ ]:
df3['properties.mp_api_endpoint'].value_counts()

In [ ]:
df3['properties.mp_lib'].value_counts()

In [ ]:
df3['properties.mp_loader'].value_counts()

In [ ]:
df3['properties.environment'].value_counts()

In [ ]:
srch = df3[df3['properties.environment'] != 'production']

In [ ]:
srch['properties.region'].value_counts()

In [ ]:
df3['properties.extension_version'].value_counts()